<a href="https://colab.research.google.com/github/Kaykayag/Brainrot-News-Advisory-Chatbot/blob/main/Brainrot_News_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 — Package installation
!pip install -q -U fastapi uvicorn nest_asyncio pyngrok requests feedparser \
    groq chromadb sentence-transformers apscheduler python-multipart \
    langchain-core langchain-chroma langchain-huggingface

print("✅ All packages installed.")

✅ All packages installed.


In [2]:
# CELL 2 — Credentials & Settings
import os
from getpass import getpass

# TODO: ============================================================
# TODO: SECRETS SETUP — read this before running this cell!
# TODO:
# TODO: Preferred method: Colab's built-in Secrets manager
# TODO: (click the key icon in the left sidebar of Colab), then:
# TODO:   1. Click "+ Add new secret"
# TODO:   2. Add EACH of these secrets using this EXACT name:
# TODO:        META_ACCESS_TOKEN     -> Meta WhatsApp Cloud API access token
# TODO:        META_PHONE_NUMBER_ID  -> "Phone number ID" from Meta App > WhatsApp > API Setup
# TODO:        META_VERIFY_TOKEN     -> Any string YOU invent (used to verify the webhook)
# TODO:        GROQ_API_KEY          -> API key from https://console.groq.com/keys
# TODO:        NGROK_AUTH_TOKEN      -> Token from https://dashboard.ngrok.com/get-started/your-authtoken
# TODO:        ADMIN_API_KEY         -> Any string YOU invent (protects /admin/captain-update)
# TODO:   3. Toggle "Notebook access" ON for each secret.
# TODO:
# TODO: If a secret isn't found in Colab Secrets or env vars, this cell falls
# TODO: back to a hidden interactive prompt (getpass). Nothing is ever printed.
# TODO: ============================================================

def _get_secret(name: str, is_sensitive: bool = True) -> str:
    """Try Colab userdata secrets first, then env var, then interactive fallback."""
    value = None
    try:
        from google.colab import userdata
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    except ImportError:
        value = None

    if not value:
        value = os.environ.get(name)

    if not value:
        prompt = f"Enter value for {name}: "
        value = getpass(prompt) if is_sensitive else input(prompt)

    if not value:
        raise ValueError(f"Missing required credential: {name}")

    return value.strip()


print("Loading credentials (Colab Secrets -> env var -> interactive prompt)...")

META_ACCESS_TOKEN    = _get_secret("META_ACCESS_TOKEN")
META_PHONE_NUMBER_ID = _get_secret("META_PHONE_NUMBER_ID")
META_VERIFY_TOKEN    = _get_secret("META_VERIFY_TOKEN")
GROQ_API_KEY         = _get_secret("GROQ_API_KEY")
NGROK_AUTH_TOKEN     = _get_secret("NGROK_AUTH_TOKEN")
ADMIN_API_KEY        = _get_secret("ADMIN_API_KEY")

# TODO: Non-secret settings — adjust to taste, none of these need to be secret
GROQ_MODEL               = "qwen/qwen3.8-27b"        # TODO: change model if Groq deprecates this one
META_GRAPH_API_VERSION   = "v19.0"                           # TODO: bump when Meta deprecates this API version
SERVER_PORT              = 8000                               # TODO: change if 8000 is already in use
OUTAGE_SCAN_INTERVAL_MIN = 15                                 # TODO: background worker scan frequency (minutes)
CHROMA_PERSIST_DIR       = "/content/chroma_db"               # TODO: change vector DB storage path if desired
USERS_DB_PATH            = "/content/registered_users.json"   # TODO: change registered-users storage path
ALERTED_CACHE_PATH       = "/content/alerted_cache.json"      # TODO: change outage-dedupe cache path

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("All credentials loaded successfully. Nothing was printed or hardcoded above.")


Loading credentials (Colab Secrets -> env var -> interactive prompt)...
All credentials loaded successfully. Nothing was printed or hardcoded above.


In [3]:
# CELL 3 — News & Advisory Aggregator
import feedparser
import urllib.parse
import datetime


def _build_gnews_rss_url(query: str, hl: str = "en-PH", gl: str = "PH", ceid: str = "PH:en") -> str:
    """Builds a Google News RSS search URL for an arbitrary query string."""
    encoded_query = urllib.parse.quote(query)
    return f"https://news.google.com/rss/search?q={encoded_query}&hl={hl}&gl={gl}&ceid={ceid}"


def _parse_feed(url: str, limit: int = 5) -> list:
    feed = feedparser.parse(url)
    articles = []
    for entry in feed.entries[:limit]:
        source_title = "Google News"
        try:
            if hasattr(entry, "source") and entry.source:
                source_title = entry.source.get("title", "Google News")
        except Exception:
            pass
        articles.append({
            "title": getattr(entry, "title", "").strip(),
            "link": getattr(entry, "link", "").strip(),
            "published": getattr(entry, "published", "") or getattr(entry, "updated", ""),
            "source": source_title,
        })
    return articles


def fetch_worldwide_news(place: str, limit: int = 5) -> list:
    """Dynamic worldwide news for ANY city/country, e.g. 'Tokyo', 'London', 'Sydney', 'New York'."""
    query = f"{place} news"
    url = _build_gnews_rss_url(query, hl="en-US", gl="US", ceid="US:en")
    return _parse_feed(url, limit)


def fetch_local_ph_news(place: str, limit: int = 5) -> list:
    """Local Philippine LGU / city / province news, e.g. 'Quezon City', 'Cebu', 'Davao'."""
    query = f"{place} Philippines news advisory"
    url = _build_gnews_rss_url(query, hl="en-PH", gl="PH", ceid="PH:en")
    return _parse_feed(url, limit)


def fetch_barangay_captain_updates(place: str, limit: int = 5) -> list:
    """Targeted search for barangay captain / kapitan / punong barangay bulletins for a specific location."""
    query = f'"{place}" ("Barangay Captain" OR "Kapitan" OR "Kap" OR "Punong Barangay")'
    url = _build_gnews_rss_url(query, hl="en-PH", gl="PH", ceid="PH:en")
    return _parse_feed(url, limit)


def fetch_outage_advisories(place: str, limit: int = 8) -> list:
    """Scans for power/water outage & emergency advisory terms for a given location."""
    query = f'"{place}" (brownout OR "power interruption" OR "water interruption" OR outage OR blackout OR advisory OR emergency)'
    url = _build_gnews_rss_url(query, hl="en-PH", gl="PH", ceid="PH:en")
    return _parse_feed(url, limit)


def format_articles_for_context(articles: list, label: str) -> str:
    if not articles:
        return f"[{label}] No recent articles found."
    lines = [f"[{label}]"]
    for a in articles:
        title = a["title"]
        source = a["source"]
        published = a["published"]
        link = a["link"]
        lines.append(f"- {title} ({source}, {published}) -> {link}")
    return "\n".join(lines)


print("News aggregator ready: fetch_worldwide_news, fetch_local_ph_news, "
      "fetch_barangay_captain_updates, fetch_outage_advisories")


News aggregator ready: fetch_worldwide_news, fetch_local_ph_news, fetch_barangay_captain_updates, fetch_outage_advisories


In [4]:
# CELL 4 — Vector Store, Embeddings, Groq LLM, Brainrot RAG pipeline
import re
import uuid

# Updated standalone imports
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq

print("Loading sentence-transformer embeddings (all-MiniLM-L6-v2)...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="news_advisory_kb",
    embedding_function=embedding_model,
    persist_directory=CHROMA_PERSIST_DIR,
)

groq_client = Groq(api_key=GROQ_API_KEY)


def ingest_document(text: str, metadata: dict = None) -> str:
    """Adds a single text chunk (news article, official bulletin, advisory) into the vector store."""
    doc_id = str(uuid.uuid4())
    doc = Document(page_content=text, metadata=metadata or {})
    vectorstore.add_documents([doc], ids=[doc_id])
    try:
        vectorstore.persist()
    except Exception:
        pass  # newer chromadb versions auto-persist
    return doc_id


def retrieve_advisories(query: str, k: int = 4) -> list:
    """Semantic search over the persistent knowledge base returning raw Document objects."""
    try:
        return vectorstore.similarity_search(query, k=k)
    except Exception as e:
        print(f"retrieve_advisories error: {e}")
        return []


# ---------------------------------------------------------------------------
# PERSONA & DETERMINISTIC OVERRIDES
# ---------------------------------------------------------------------------
VOCAB_SHOCK_WORDS = [
    "extravagant", "flabbergasted", "quintessential", "incomprehensible",
    "unequivocal", "ubiquitous", "esoteric", "perfunctory", "cacophony",
    "magnanimous", "surreptitious", "juxtaposition", "ephemeral",
]

PWEDE_MAGTANONG_PATTERN = re.compile(r"pwede\s*mag\s*tanong\??", re.IGNORECASE)
CAN_I_ASK_PATTERN = re.compile(r"\bcan\s*i\s*ask\??", re.IGNORECASE)
PWEDE_MAGTANONG_REPLY = "never grow old? Huyyy bawal mag-inarte, drop the tea accla!"


def _check_deterministic_overrides(user_text: str) -> str:
    """Returns a forced prefix string if a hard-rule trigger fires, else an empty string."""
    prefix = ""
    if PWEDE_MAGTANONG_PATTERN.search(user_text) or CAN_I_ASK_PATTERN.search(user_text):
        prefix += PWEDE_MAGTANONG_REPLY + "\n\n"

    lowered = user_text.lower()
    for word in VOCAB_SHOCK_WORDS:
        if word in lowered:
            prefix += f"{word.upper()}??! Over naman sa pa-vocab teh, sino bang sinasaktan mo?!\n\n"
            break
    return prefix


def generate_brainrot_response(user_query: str, sender_phone: str = None, location_hint: str = None) -> str:
    # 1. Deterministic trigger check (handles standalone "pwede magtanong")
    cleaned_query = user_query.strip().lower()
    if cleaned_query in ["pwede magtanong", "pwede magtanong?", "pwede mag tanong?"]:
        return PWEDE_MAGTANONG_REPLY

    # Check for prefix overrides (if the user asked "pwede magtanong may brownout ba?")
    prefix_override = _check_deterministic_overrides(user_query)

    # 2. Resolve registered location
    user_location = location_hint
    if not user_location and sender_phone:
        user_location = registered_users.get(sender_phone, {}).get("location")

    # 3. Retrieve context from ChromaDB
    search_term = f"{user_query} {user_location}" if user_location else user_query
    docs = retrieve_advisories(search_term)

    context_text = "\n\n".join([d.page_content for d in docs]) if docs else "No official advisory found."

    # 4. Location context anchoring
    location_instruction = (
        f"The user has permanently locked in their home base as: '{user_location}'. "
        f"Answer their questions assuming they are asking specifically about '{user_location}'."
        if user_location else
        "The user has no registered location yet. You can remind them to send 'set location <Barangay/City>'."
    )

    system_prompt = f"""You are an energetic, witty Gen Z Pinoy AI assistant with internet 'brainrot' slang (Taglish: 'accla', 'no cap', 'solulu', 'fr fr', 'bombastic side eye').
{location_instruction}

Context from advisories/news:
{context_text}

Rules:
- Give a brief, helpful answer grounded in the context.
- If there are no outages/bulletins found, tell them things look peaceful in their area in your persona.
"""

    try:
        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7,
            max_tokens=300
        )
        ai_reply = completion.choices[0].message.content.strip()
        return f"{prefix_override}{ai_reply}".strip()
    except Exception as e:
        return (f"Ay teh, medyo nag-lag ang sistema ko ({e}). "
                f"Try mo ulit in a bit, promise solulu na 'to!")

Loading sentence-transformer embeddings (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
# CELL 5 — Outbound dispatcher + background worker
import requests
import json
import hashlib
from apscheduler.schedulers.background import BackgroundScheduler

META_SEND_URL = f"https://graph.facebook.com/{META_GRAPH_API_VERSION}/{META_PHONE_NUMBER_ID}/messages"


def send_meta_whatsapp(to: str, message: str) -> dict:
    """Sends a WhatsApp text message via the official Meta Cloud API."""
    headers = {
        "Authorization": f"Bearer {META_ACCESS_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": to,
        "type": "text",
        "text": {"preview_url": False, "body": message},
    }
    resp = requests.post(META_SEND_URL, headers=headers, json=payload, timeout=15)
    if resp.status_code >= 400:
        print(f"Meta send error [{resp.status_code}]: {resp.text}")
    return {"status_code": resp.status_code, "body": resp.text}


# ---------------------------------------------------------------------------
# Registered-user store: phone -> {"location": str, "registered_at": iso}
# Persisted as JSON so it survives across cell re-runs within the same runtime.
# ---------------------------------------------------------------------------
def _load_json(path: str, default):
    if os.path.exists(path):
        try:
            with open(path, "r") as f:
                return json.load(f)
        except Exception:
            return default
    return default


def _save_json(path: str, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


registered_users = _load_json(USERS_DB_PATH, {})
alerted_cache = _load_json(ALERTED_CACHE_PATH, {})  # phone -> [article_hash, ...]


def register_user_location(phone: str, location: str) -> None:
    registered_users[phone] = {
        "location": location.strip(),
        "registered_at": datetime.datetime.utcnow().isoformat(),
    }
    _save_json(USERS_DB_PATH, registered_users)


def _article_hash(article: dict) -> str:
    return hashlib.sha256((article["title"] + article["link"]).encode("utf-8")).hexdigest()


OUTAGE_KEYWORDS = ["brownout", "blackout", "power interruption", "water interruption",
                    "outage", "emergency", "advisory"]


def check_outage_alerts():
    """Runs every OUTAGE_SCAN_INTERVAL_MIN minutes. Scans each registered user's saved
    location for outage/emergency advisories and pushes an UNSOLICITED WhatsApp alert
    for anything new."""
    print(f"[{datetime.datetime.now().isoformat()}] Scanning outage advisories for "
          f"{len(registered_users)} registered user(s)...")

    for phone, info in list(registered_users.items()):
        location = info.get("location")
        if not location:
            continue

        try:
            articles = fetch_outage_advisories(location, limit=8)
        except Exception as e:
            print(f"Feed error for {phone} ({location}): {e}")
            continue

        seen_hashes = set(alerted_cache.get(phone, []))
        new_alerts = []

        for a in articles:
            text_blob = (a["title"] + " " + location).lower()
            if not any(k in text_blob for k in OUTAGE_KEYWORDS):
                continue
            h = _article_hash(a)
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
            new_alerts.append(a)

        if new_alerts:
            alerted_cache[phone] = list(seen_hashes)[-200:]  # cap cache size
            _save_json(ALERTED_CACHE_PATH, alerted_cache)

            lines = [
                f"Bombastic side eye alert, accla! May outage/advisory update sa "
                f"*{location}* -- official Brownout Arc loading:",
                "",
            ]
            for a in new_alerts[:3]:
                title = a["title"]
                source = a["source"]
                link = a["link"]
                lines.append(f"- {title} ({source})\n  {link}")
            lines.append("")
            lines.append("Delulu is hoping wala 'to, SOLULU is charging your gadgets/powerbank "
                          "AND filling water containers NOW. Di dasurv ang stranded sa dilim, teh!")

            alert_message = "\n".join(lines)
            send_meta_whatsapp(phone, alert_message)
            print(f"Pushed unsolicited alert to {phone} for {location}")

    print("Outage scan complete.")


scheduler = BackgroundScheduler()
scheduler.add_job(check_outage_alerts, "interval", minutes=OUTAGE_SCAN_INTERVAL_MIN,
                   id="outage_scan_job", replace_existing=True)


def start_background_worker():
    if not scheduler.running:
        scheduler.start()
        print(f"Background worker started -- scanning every {OUTAGE_SCAN_INTERVAL_MIN} minutes.")
    else:
        print("Background worker already running.")


print("Dispatcher + scheduler ready. start_background_worker() is auto-called in Cell 7.")


Dispatcher + scheduler ready. start_background_worker() is auto-called in Cell 7.


In [6]:
# CELL 6 — FastAPI application
import re
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import PlainTextResponse, JSONResponse

app = FastAPI(title="WhatsApp News & Advisory Chatbot")


@app.get("/webhook")
async def verify_webhook(request: Request):
    """Meta's Hub challenge verification handshake."""
    params = request.query_params
    mode = params.get("hub.mode")
    token = params.get("hub.verify_token")
    challenge = params.get("hub.challenge")

    if mode == "subscribe" and token == META_VERIFY_TOKEN:
        print("Webhook verified by Meta.")
        return PlainTextResponse(content=challenge, status_code=200)

    print("Webhook verification failed -- token mismatch.")
    raise HTTPException(status_code=403, detail="Verification failed")


LOCATION_COMMAND_PATTERN = re.compile(r"^(set\s*location|subscribe)\s*[:\-]?\s*(.+)$", re.IGNORECASE)


@app.post("/webhook")
async def receive_message(request: Request):
    """Handles inbound WhatsApp messages from the Meta Cloud API."""
    data = await request.json()

    try:
        entry = data["entry"][0]
        change = entry["changes"][0]
        value = change["value"]

        if "messages" not in value:
            # Delivery receipts / read statuses / other webhook events -- ignore cleanly.
            return JSONResponse(content={"status": "ignored_non_message_event"}, status_code=200)

        message = value["messages"][0]

        if message.get("type") != "text":
            # Ignore images, audio, location pins, reactions, stickers, etc.
            return JSONResponse(content={"status": "ignored_non_text_message"}, status_code=200)

        sender = message["from"]
        text_body = message["text"]["body"].strip()

        print(f"Inbound from {sender}: {text_body}")

       # Inside receive_message in Cell 6:
        loc_match = LOCATION_COMMAND_PATTERN.match(text_body)
        if loc_match:
            location = loc_match.group(2).strip()
            register_user_location(sender, location)
            reply = (f"Naka-lock in na, accla! Sini-save ko na *{location}* as your spot. "
                     f"Automatic na akong mag-aalert sa'yo pag may brownout/water interruption "
                     f"dyan, no cap. Real ba na ready ka na? Fr fr, solulu mode ON.")
        else:
            # Look up saved location from the registered dictionary
            saved_location = registered_users.get(sender, {}).get("location")
            reply = generate_brainrot_response(text_body, sender_phone=sender, location_hint=saved_location)

        send_meta_whatsapp(sender, reply)
        return JSONResponse(content={"status": "ok"}, status_code=200)

    except (KeyError, IndexError, TypeError) as e:
        print(f"Non-message payload received or parse issue (safe to ignore): {e}")
        return JSONResponse(content={"status": "ignored_unparseable_event"}, status_code=200)
    except Exception as e:
        print(f"Unexpected error handling inbound message: {e}")
        return JSONResponse(content={"status": "error", "detail": str(e)}, status_code=200)


@app.post("/admin/captain-update")
async def captain_update(request: Request, x_admin_key: str = Header(None)):
    """Protected admin route to inject official barangay-captain bulletins into the KB.

    Call with header 'X-Admin-Key: <ADMIN_API_KEY>' and JSON body:
    {"location": "Barangay San Isidro, Quezon City", "message": "Water interruption on..."}
    """
    if x_admin_key != ADMIN_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid admin key")

    body = await request.json()
    location = body.get("location")
    message = body.get("message")

    if not location or not message:
        raise HTTPException(status_code=400, detail="Both 'location' and 'message' are required")

    doc_text = f"[OFFICIAL BULLETIN - {location}] {message}"
    doc_id = ingest_document(doc_text, metadata={"source": "barangay_captain", "location": location})

    print(f"Admin bulletin ingested for {location}: {message[:80]}...")
    return JSONResponse(content={"status": "ingested", "doc_id": doc_id}, status_code=200)


@app.get("/")
async def health_check():
    return {"status": "alive", "service": "WhatsApp News & Advisory Chatbot"}


print("FastAPI app defined: GET/POST /webhook, POST /admin/captain-update, GET /")


FastAPI app defined: GET/POST /webhook, POST /admin/captain-update, GET /


In [ ]:
# CELL 7 — ngrok tunnel + Uvicorn server
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf

nest_asyncio.apply()

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Close any stray tunnels left over from a previous run
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_tunnel = ngrok.connect(SERVER_PORT, "http")
public_url = public_tunnel.public_url

start_background_worker()

print("=" * 70)
print("YOUR CHATBOT IS LIVE")
print("=" * 70)
print(f"Webhook Callback URL : {public_url}/webhook")
print(f"Verify Token          : {META_VERIFY_TOKEN}")
print(f"Admin bulletin URL    : {public_url}/admin/captain-update  (header X-Admin-Key)")
print("=" * 70)
print("Paste the Callback URL + Verify Token into:")
print("Meta for Developers > Your App > WhatsApp > Configuration > Webhook")
print("Then subscribe to the 'messages' webhook field.")
print("=" * 70)

# FIX: Run Uvicorn directly on Colab's existing event loop via await
config = uvicorn.Config(app=app, host="0.0.0.0", port=SERVER_PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [9683]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Background worker started -- scanning every 15 minutes.
YOUR CHATBOT IS LIVE
Webhook Callback URL : https://moneybags-barterer-catcall.ngrok-free.dev/webhook
Verify Token          : my_token_for_meta_12345
Admin bulletin URL    : https://moneybags-barterer-catcall.ngrok-free.dev/admin/captain-update  (header X-Admin-Key)
Paste the Callback URL + Verify Token into:
Meta for Developers > Your App > WhatsApp > Configuration > Webhook
Then subscribe to the 'messages' webhook field.
Inbound from 639089827175: ano latest chika sa barangay namin
INFO:     2a03:2880:2ff:71:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:5f:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:2:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:51:::0 - "POST /webhook HTTP/1.1" 200 OK
Inbound from 639089827175: ano latest post ni barangay kap
INFO:     2a03:2880:2ff:71:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:47:::0 - "POST /webhook HTTP/1.1" 200 OK
INF

/tmp/ipykernel_9683/3244426091.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "registered_at": datetime.datetime.utcnow().isoformat(),


INFO:     2a03:2880:22ff:54:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:54:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:58:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:42:::0 - "POST /webhook HTTP/1.1" 200 OK
Inbound from 639089827175: ano latest news
INFO:     2a03:2880:2ff:5:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:70:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:5a:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:74:::0 - "POST /webhook HTTP/1.1" 200 OK
Inbound from 639089827175: i mean balita sa lugar, not just storms, chaos or what but literally news
INFO:     2a03:2880:10ff:9:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:4d:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:4e:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:5a:::0 - "POST /webhook HTTP/1.1" 200 OK
